# 13.3 Iterating subject to a condition: the repeat statement

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

A second kind of looping construct, the repeat statement, continues iterating as
long as some logical condition is satisfied.

Returning to the sensitivity analysis example, we wish to take advantage of a property
of the dual value on the constraint Time[3]: the additional profit that can be realized
from each extra hour added to avail[3] is at most Time[3].dual. When
avail[3] is made sufficiently large, so that there is more third-week capacity than can
be used, the associated dual value falls to zero and further increases in avail[3] have
no effect on the optimal solution.

We can specify that looping should stop once the dual value falls to zero, by writing a
repeat statement that has one of the following forms:

```ampl
repeat   while   Time[3].dual > 0 { . . . };
repeat   until   Time[3].dual = 0 { . . . };
repeat   {...}   while Time[3].dual > 0;
repeat   {...}   until Time[3].dual = 0;
```

The loop body, here indicated by {...}, must be enclosed in braces. Passes through the
loop continue as long as the while condition is true, or as long as the until condition
is false. A condition that appears before the loop body is tested before every pass; if a
while condition is false or an until condition is true before the first pass, then the
loop body is never executed. A condition that appears after the loop body is tested after
every pass, so that the loop body is executed at least once in this case. If there is no
while or until condition, the loop repeats indefinitely and must be terminated by
other means, like the break statement described below.

A complete script using repeat is shown in [Figure 13-3](./13_3_iterating_subject_to_a_condition_the_repeat_statement.ipynb#fig-13-3). For this particular application
we choose the until phrase that is placed after the loop body, as we do not want
Time[3].dual to be tested until after a `solve` has been executed in the first pass.
Two other features of this script are worth noting, as they are relevant to many scripts of
this kind.

At the beginning of the script, we don't know how many passes the repeat statement
will make through the loop. Thus we cannot determine AVAIL3 in advance as we
did in [Figure 13-1](./13_2_iterating_over_a_set_the_for_statement.ipynb#fig-13-1). Instead, we declare it initially to be the empty set:

```ampl
set AVAIL3 default {};
param avail3_obj {AVAIL3};
param avail3_dual {AVAIL3};
```

and add each new value of avail[3] to it after solving:

```ampl
model steelT.mod;
data steelT.dat;
option solution_precision 10;
option solver_msg 0;
set AVAIL3 default {};
param avail3_obj {AVAIL3};
param avail3_dual {AVAIL3};
param avail3_step := 5;
repeat {
       solve;
       let AVAIL3 := AVAIL3 union {avail[3]};
       let avail3_obj[avail[3]] := Total_Profit;
       let avail3_dual[avail[3]] := Time[3].dual;
       let avail[3] := avail[3] + avail3_step;
} until Time[3].dual = 0;
display avail3_obj, avail3_dual;
```

**[Figure 13-3](./13_3_iterating_subject_to_a_condition_the_repeat_statement.ipynb#fig-13-3):** Script for recording sensitivity (steelT.sa4).

```ampl
let AVAIL3 := AVAIL3 union {avail[3]};
let avail3_obj[avail[3]] := Total_Profit;
let avail3_dual[avail[3]] := Time[3].dual;
```

By adding a new member to AVAIL3, we also create new components of the parameters
avail3_obj and avail3_dual that are indexed over AVAIL3, and so we can proceed
to assign the appropriate values to these components. Any change to a set is propagated
to all declarations that use the set, in the same way that any change to a parameter
is propagated.

Because numbers in the computer are represented with a limited number of bits of
precision, a solver may return values that differ very slightly from the solution that would
be computed using exact arithmetic. Ordinarily you don't see this, because the `display`
command rounds values to six significant digits by default. For example:

```ampl
ampl: model steelT.mod; data steelT.dat; solve;
ampl: display Make;
Make [*,*] (tr)
: bands coils       :=
1   5990   1407
2   6000   1400
3   1400   3500
4   2000   4200
;
```

Compare what is shown when rounding is dropped, by setting display_precision
to 0:

```ampl
ampl: option display_precision 0;
ampl: display Make;
Make [*,*] (tr)
:         bands                coils                              :=
1   5989.999999999999    1407.0000000000002
2   6000                 1399.9999999999998
3   1399.9999999999995   3500
4   1999.9999999999993   4200
;
```

These seemingly tiny differences can have undesirable effects whenever a script makes a
comparison that uses values returned by the solver. The rounded `table` would lead you to
believe that Make["coils",2] >= 1400 is true, for example, whereas from the second
`table` you can see that really it is false.

You can avoid this kind of surprise by writing arithmetic tests more carefully; instead
of until Time[3].dual = 0, for instance, you might say until Time[3].dual
<= 0.0000001. Alternatively, you can round all solution values that are returned by
the solver, so that numbers that are supposed to be equal really do come out equal. The
statement

```ampl
option solution_precision 10;
```

toward the beginning of [Figure 13-3](./13_3_iterating_subject_to_a_condition_the_repeat_statement.ipynb#fig-13-3) has this effect; it states that solution values are to be
rounded to 10 significant digits. This and related rounding options are discussed and
illustrated in [Section 12.3](../12/12_3_numeric_options_for_display.ipynb).

Note finally that the script declares set AVAIL3 as default {} rather than = {}.
The former allows AVAIL3 to be changed by `let` commands as the script proceeds,
whereas the latter permanently defines AVAIL3 to be the empty set.